# Lesson 06 — pandas for Handling Data

**What this notebook does:** builds a small table of support tickets using pandas, then loads it, looks at it, selects pieces of it, filters rows, and cleans a missing value — the four things you do to almost every real dataset before any model ever sees it.

## Step 1 — Import pandas

The convention almost everyone uses is `import pandas as pd` — same idea as `np` for NumPy last lesson, just a short nickname. pandas is built directly on top of NumPy: underneath, a pandas table stores its numbers in NumPy arrays. pandas adds two things NumPy does not have on its own: labels (column names, row labels) and the ability to mix column types in one table (numbers in one column, text in another).

In [ ]:
import pandas as pd

## Step 2 — Building a table: the DataFrame

pandas' main object is called a **DataFrame** — a table of rows and columns, like a spreadsheet, where every column can hold its own type (text in one column, whole numbers in another, decimals in a third). This is different from a NumPy array, where every single element in the whole array must be the same type. Each column, pulled out on its own, is called a **Series** — think of it as one labeled column, or one row, of the table.

The easiest way to build a small DataFrame by hand is from a Python dictionary, where each key becomes a column name and each value is a list of that column's entries, one per row. Below are six support tickets: an id, the ticket text, its category, and a 1-to-5 customer satisfaction rating taken *after* the ticket was resolved — one ticket has no rating yet, written as `None`, because that customer has not answered the survey.

Every row also gets a label, shown on the left when the table prints. Unless told otherwise, pandas numbers rows `0, 1, 2, ...` automatically — this row-label column is called the **index**.

In [ ]:
tickets = {
    "ticket_id": [1, 2, 3, 4, 5, 6],
    "text": [
        "My payment failed twice and I need this resolved right now!",
        "Do you have this item in a larger size?",
        "Where is my order, it has not arrived yet!!",
        "Thanks for the quick delivery, love it!",
        "The app keeps crashing when I try to check out",
        "Can I get a refund for a damaged item?",
    ],
    "category": ["billing", "product", "shipping", "shipping", "product", "billing"],
    "satisfaction_rating": [2, 4, 1, 5, None, 3],
}

df = pd.DataFrame(tickets)
print(df)

## Step 3 — Looking at the data: `.head()`, `.shape`, `.info()`

Before doing anything to a real dataset, look at it first. Three quick tools:

- `.head(n)` shows just the first `n` rows — useful when a table has thousands of rows and printing all of them would flood the screen.
- `.shape` — the exact same idea as a NumPy array's shape from last lesson — gives `(rows, columns)` as a tuple.
- `.info()` prints a summary of every column: its name, how many non-missing ("non-null") values it has, and its dtype (data type). `.info()` already prints its own output directly, so it is called on its own line, not wrapped in `print(...)`.

Watch the `satisfaction_rating` column's dtype. We put in whole numbers and one `None`. NumPy (which pandas uses underneath) has no way to put "missing" inside a column of whole numbers (an `int` column) — there is no integer value that means "nothing here." So pandas quietly upgrades the entire column to decimals (`float64`) instead, because floating-point numbers do have a special "missing number" value, written `NaN` ("Not a Number"). This is why `2` shows up as `2.0` — the whole column became decimals just to make room for the one missing entry.

In [ ]:
print(df.head(3))
print(df.shape)
df.info()

## Step 4 — Selecting columns: one bracket versus two

`df['category']` pulls out a single column as a Series — a labeled 1D array, one value per row, carrying the original row index along with it.

`df[['text', 'category']]` — note the *double* brackets — pulls out several columns at once as a smaller DataFrame. The outer bracket means "select columns from `df`"; the inner bracket is a plain Python list of the column names wanted. A single name with no list, `df['category']`, hands back a Series; a list of names, even a list of just one (`df[['category']]`), hands back a DataFrame. The number of brackets is the whole difference between the two.

In [ ]:
print(df["category"])
print(type(df["category"]))

print(df[["text", "category"]])
print(type(df[["text", "category"]]))

## Step 5 — Selecting rows: `.loc` versus `.iloc`

There are two ways to grab a specific row, and they answer two different questions:

- `.iloc[i]` — "**i**nteger **loc**ation" — grabs the row at position `i`, counting from the top, exactly like indexing into a plain Python list. It never looks at the row's label, only its position.
- `.loc[label]` — grabs the row whose index label *is* `label`, wherever it happens to sit in the table.

Right now these look identical, because our DataFrame still has the plain default index (`0, 1, 2, 3, 4, 5`), where position and label happen to be the same number. They stop matching the moment the index becomes anything else — for example, after filtering rows out (a row's label stays attached to it, but its position shifts), or after setting a meaningful column, like `ticket_id`, as the index. Both tools exist because "the 3rd row" and "the row labeled 3" are genuinely different questions once a table has been filtered, sorted, or given a custom index.

In [ ]:
print(df.iloc[0])
print(df.loc[0])

## Step 6 — Filtering rows with a boolean mask

This is the exact same pattern as Lesson 05's `np_lengths > 20`, just applied to a table instead of a flat array. `df['category'] == 'shipping'` compares every row's category to `'shipping'` at once and returns a Series of `True`/`False` — a boolean mask, one entry per row this time instead of one per number. `df[mask]` then keeps only the rows where the mask is `True`. No loop, no `if`, written by us — build the mask with a comparison, then index with it.

In [ ]:
mask = df["category"] == "shipping"
print(mask)
print(df[mask])

## Step 7 — Missing values: finding, filling, and dropping them

Real data almost always has gaps — here, one customer never rated their resolved ticket. Three tools handle this:

- `.isna()` — the same boolean-mask idea again — returns `True` wherever a value is missing (`NaN`) and `False` everywhere else.
- `.fillna(value)` replaces every missing entry with `value`. Guessing `0` would be misleading here (it would look like an angry 1-star-below-worst rating that never happened), so instead we fill with the column's own `.mean()` — the exact same aggregate function from Lesson 05 — a reasonable, neutral stand-in for "we don't actually know."
- `.dropna()` removes every row that has *any* missing value at all, gap and all.

One easy-to-miss detail: `.fillna(...)` and `.dropna()` do not change `df` itself. Like most pandas methods, they hand back a **brand-new** table (or Series) with the change applied, leaving the original untouched — unless the result is reassigned (`df = df.dropna()`) or `inplace=True` is passed. That is why `df.shape` still shows all 6 rows even after calling `.dropna()` below.

In [ ]:
print(df["satisfaction_rating"])
print(df["satisfaction_rating"].isna())

average_rating = df["satisfaction_rating"].mean()
filled_ratings = df["satisfaction_rating"].fillna(average_rating)
print(filled_ratings)

print(df.dropna())
print(df.shape)

## Step 8 — Adding a new column with vectorized string operations

pandas' `.str` accessor unlocks string operations that run across an entire text column at once — the same vectorization idea from Lesson 05, now applied to text instead of numbers. `df['text'].str.len()` computes the character length of every ticket in one call; `df['text'].str.count('!')` counts how many `!` characters are in every ticket. No loop written by us, same as `np_lengths * 2` last lesson.

Assigning either result to a new column name, `df['length'] = ...`, adds a brand-new column to the table — this is exactly the two-feature table (length and exclamation count) sketched by hand back in Lesson 05, now built for real from actual ticket text instead of typed in by hand.

In [ ]:
df["length"] = df["text"].str.len()
df["exclamations"] = df["text"].str.count("!")

print(df[["text", "length", "exclamations"]])
print(df.shape)